# Construction of Environmental Features

This notebook extracts environmental and socioeconomic predictors for the 4,985 grid cells containing LTE or 5G NR labels.

Google Earth Engine is used to derive Sentinel-1 backscatter, Sentinel-2 spectral indices, ESA WorldCover land-cover proportions, WorldPop population, and VIIRS night-time-light intensity.

The extracted features are merged with the LTE and 5G NR labels using the unique 'grid_id'.

※ Related dissertation sections
 - Section 2.6 (Public Geospatial Data as Predictors)
- Section 3.1 (Research Approach—Data Understanding and Data Preparation)
 - Section 4.1.1 (Baseline Dataset Construction).

In [ ]:
# ============================================================
# Step 1. Mount Google Drive
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Environment and Input Data

This section mounts Google Drive, imports the required Python libraries, authenticates Google Earth Engine, and loads the England grid and LTE and 5G NR labels created in the previous notebook.

Only grid cells containing at least one LTE or 5G NR label are retained for environmental-feature extraction.

※Related dissertation sections
 - Section 3.1 (Research Approach—Data Understanding and Data Preparation)
 - Appendix C (Reproducibility and Predictor Definitions).

In [ ]:
# ============================================================
# Step 2. Import Libraries
# ============================================================

import os
import json
import numpy as np
import pandas as pd
import geopandas as gpd

In [ ]:
# ============================================================
# Step 3. Initialize Google Earth Engine
# ============================================================

import ee

ee.Authenticate()
ee.Initialize(project="cw-project-479516")

print("GEE initialized.")

GEE initialized.


In [ ]:
# ============================================================
# Step 4. Load Dataset19 Labels and England Grid
# ============================================================

base_dir = (
    "/content/drive/MyDrive/Dissertation/Experiments/"
    "최종실험(수정)/Dataset_Build"
)

# Load LTE labels
lte = pd.read_csv(
    os.path.join(
        base_dir,
        "01_Ofcom",
        "dataset19_lte_labels_final.csv"
    )
)

# Load 5G NR labels
nr = pd.read_csv(
    os.path.join(
        base_dir,
        "01_Ofcom",
        "dataset19_nr_labels_final.csv"
    )
)

# Load the new England 1 km grid
grid = gpd.read_file(
    os.path.join(
        base_dir,
        "00_Grid",
        "england_1km_grid.gpkg"
    ),
    layer="england_1km_grid"
)

print("LTE labels :", lte.shape)
print("5G NR labels:", nr.shape)
print("England grid:", grid.shape)
print("Grid CRS    :", grid.crs)

LTE labels : (4970, 6)
5G NR labels: (4965, 6)
England grid: (133440, 2)
Grid CRS    : EPSG:27700


## 2. Sentinel-1 Radar Features

Sentinel-1 Ground Range Detected imagery collected between August and October 2025 is used to calculate median VV and VH radar backscatter.

The resulting values provide information about surface structure, vegetation, moisture, and built environments. Mean VV and VH values are extracted for each labelled 1 km grid cell using Google Earth Engine.

### ※ Related dissertation sections

- Section 2.6 (Public Geospatial Data as Predictors)
- Section 3.1 (Research Approach—Data Preparation)
- Section 4.1.1 (Baseline Dataset Construction)

In [ ]:
# ============================================================
# Step 5. Prepare Sentinel-1
# ============================================================

s1 = (
    ee.ImageCollection("COPERNICUS/S1_GRD")
    .filterDate("2025-08-01", "2025-11-01")
    .filterBounds(ee.Geometry.Rectangle([-6.0, 49.8, 2.2, 55.9]))
    .filter(ee.Filter.eq("instrumentMode", "IW"))
    .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
    .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
    .select(["VV", "VH"])
)

s1_median = s1.median()

print("Sentinel-1 images:", s1.size().getInfo())

Sentinel-1 images: 978


In [ ]:
# ============================================================
# Step 6. Select Grids with LTE or 5G NR Labels
# ============================================================

# Combine labelled grid IDs from LTE and 5G NR
labelled_grid_ids = set(lte["grid_id"]).union(
    set(nr["grid_id"])
)

# Select only labelled grids
labelled_grid = grid[
    grid["grid_id"].isin(labelled_grid_ids)
].copy()

labelled_grid.reset_index(drop=True, inplace=True)

# Validate selection
print("LTE labelled grids       :", f"{lte['grid_id'].nunique():,}")
print("5G NR labelled grids     :", f"{nr['grid_id'].nunique():,}")
print("Combined labelled grids  :", f"{len(labelled_grid):,}")
print(
    "LTE–NR overlapping grids:",
    f"{len(set(lte['grid_id']) & set(nr['grid_id'])):,}"
)
print("CRS:", labelled_grid.crs)

LTE labelled grids       : 4,970
5G NR labelled grids     : 4,965
Combined labelled grids  : 4,985
LTE–NR overlapping grids: 4,950
CRS: EPSG:27700


In [ ]:
# ============================================================
# Step 7. Export Sentinel-1 (VV and VH) by Batch
# ============================================================

# Convert labelled grids to WGS84 for Google Earth Engine
grid_wgs84 = labelled_grid.to_crs(epsg=4326)

batch_size = 500
n = len(grid_wgs84)

for start in range(0, n, batch_size):

    end = min(start + batch_size, n)

    # Convert current batch to GeoJSON
    batch_gdf = grid_wgs84.iloc[start:end][
        ["grid_id", "geometry"]
    ].copy()

    batch_geojson = json.loads(
        batch_gdf.to_json()
    )

    grids_batch = ee.FeatureCollection(batch_geojson)

    # Calculate mean VV and VH for each grid
    result = s1_median.reduceRegions(
        collection=grids_batch,
        reducer=ee.Reducer.mean(),
        scale=100,
        tileScale=16
    ).select(
        ["grid_id", "VV", "VH"]
    )

    # Export results to Google Drive
    task = ee.batch.Export.table.toDrive(
        collection=result,
        description=f"dataset19_sentinel1_batch_{start}_{end}",
        folder="Dataset19_Sentinel1_Batches",
        fileNamePrefix=f"dataset19_sentinel1_batch_{start}_{end}",
        fileFormat="CSV"
    )

    task.start()

    print(f"Started batch {start:,}-{end:,}")

print("\nAll 10 Sentinel-1 export tasks submitted.")

Started batch 0-500
Started batch 500-1,000
Started batch 1,000-1,500
Started batch 1,500-2,000
Started batch 2,000-2,500
Started batch 2,500-3,000
Started batch 3,000-3,500
Started batch 3,500-4,000
Started batch 4,000-4,500
Started batch 4,500-4,985

All 10 Sentinel-1 export tasks submitted.


## 3. Sentinel-2 Spectral Features

Sentinel-2 surface-reflectance imagery collected between August and October 2025 is filtered and masked to reduce contamination from clouds, cloud shadows, cirrus, and snow.

Two spectral indices are calculated:

- NDVI: represents vegetation characteristics.
- NDBI: represents built-up surface characteristics.

Mean NDVI and NDBI values are extracted for each labelled 1 km grid cell.

※Related dissertation sections
 - Section 2.6 (Public Geospatial Data as Predictors),
 - Section 3.1 (Research Approach—Data Preparation),
 - Section 4.1.1 (Baseline Dataset Construction).

In [ ]:
# ============================================================
# Step 8. Prepare Sentinel-2 (NDVI and NDBI)
# ============================================================

region = ee.Geometry.Rectangle([-6.0, 49.8, 2.2, 55.9])

# Mask cloud shadow, clouds, cirrus, and snow using SCL
def mask_s2_clouds(image):
    scl = image.select("SCL")

    clear_mask = (
        scl.neq(3)   # Cloud shadow
        .And(scl.neq(8))   # Medium-probability cloud
        .And(scl.neq(9))   # High-probability cloud
        .And(scl.neq(10))  # Cirrus
        .And(scl.neq(11))  # Snow or ice
    )

    return image.updateMask(clear_mask)

s2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterDate("2025-08-01", "2025-11-01")
    .filterBounds(region)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 50))
    .map(mask_s2_clouds)
)

s2_median = s2.median()

ndvi = s2_median.normalizedDifference(
    ["B8", "B4"]
).rename("NDVI")

ndbi = s2_median.normalizedDifference(
    ["B11", "B8"]
).rename("NDBI")

s2_features = ndvi.addBands(ndbi)

print("Sentinel-2 images:", s2.size().getInfo())

Sentinel-2 images: 1502


In [ ]:
# ============================================================
# Step 9. Export Sentinel-2 (NDVI and NDBI) by Batch
# ============================================================

batch_size = 500
n = len(grid_wgs84)

for start in range(0, n, batch_size):

    end = min(start + batch_size, n)

    batch_gdf = grid_wgs84.iloc[start:end][
        ["grid_id", "geometry"]
    ].copy()

    batch_geojson = json.loads(
        batch_gdf.to_json()
    )

    grids_batch = ee.FeatureCollection(batch_geojson)

    result = s2_features.reduceRegions(
        collection=grids_batch,
        reducer=ee.Reducer.mean(),
        scale=20,
        tileScale=16
    ).select(
        ["grid_id", "NDVI", "NDBI"]
    )

    task = ee.batch.Export.table.toDrive(
        collection=result,
        description=f"dataset19_sentinel2_batch_{start}_{end}",
        folder="Dataset19_Sentinel2_Batches",
        fileNamePrefix=f"dataset19_sentinel2_batch_{start}_{end}",
        fileFormat="CSV"
    )

    task.start()

    print(f"Started batch {start:,}-{end:,}")

print("\nAll 10 Sentinel-2 export tasks submitted.")

Started batch 0-500
Started batch 500-1,000
Started batch 1,000-1,500
Started batch 1,500-2,000
Started batch 2,000-2,500
Started batch 2,500-3,000
Started batch 3,000-3,500
Started batch 3,500-4,000
Started batch 4,000-4,500
Started batch 4,500-4,985

All 10 Sentinel-2 export tasks submitted.


## 4. Merge Sentinel Features

The batch-level Sentinel-1 and Sentinel-2 outputs are combined and validated using 'grid_id'.

The resulting VV, VH, NDVI, and NDBI features are then merged with the LTE and 5G NR labels to create an intermediate environmental dataset.

※ Related dissertation sections
 - Section 3.1 (Research Approach—Data Preparation)
 - Section 4.1.1 (Baseline Dataset Construction).

In [ ]:
# ============================================================
# Step 10. Merge Sentinel-1 Batch CSV Files
# ============================================================

import glob

s1_batch_dir = (
    "/content/drive/MyDrive/"
    "Dataset19_Sentinel1_Batches (1)"
)

s1_files = sorted(
    glob.glob(
        os.path.join(
            s1_batch_dir,
            "dataset19_sentinel1_batch_*.csv"
        )
    )
)

print("Sentinel-1 files:", len(s1_files))

if len(s1_files) != 10:
    raise ValueError(
        f"Expected 10 Sentinel-1 files, but found {len(s1_files)}."
    )

s1_df = pd.concat(
    [pd.read_csv(file) for file in s1_files],
    ignore_index=True
)

s1_df = s1_df[
    ["grid_id", "VV", "VH"]
].copy()

if s1_df["grid_id"].duplicated().any():
    raise ValueError("Duplicate grid IDs found in Sentinel-1 data.")

print("Sentinel-1 shape:", s1_df.shape)
print("Unique grid IDs:", s1_df["grid_id"].nunique())
print("Missing VV:", s1_df["VV"].isna().sum())
print("Missing VH:", s1_df["VH"].isna().sum())

display(s1_df.head())

Sentinel-1 files: 10
Sentinel-1 shape: (4985, 3)
Unique grid IDs: 4985
Missing VV: 0
Missing VH: 0


,grid_id,VV,VH
0,3975,-8.570331,-17.172422
1,3976,-7.236509,-16.062169
2,4072,-7.423318,-15.959686
3,4166,-10.591515,-17.102621
4,4167,-12.357060,-19.367379


In [ ]:
# ============================================================
# Step 11. Merge Sentinel-2 Batch CSV Files
# ============================================================

s2_files = sorted(
    glob.glob(
        "/content/drive/MyDrive/"
        "Dataset19_Sentinel2_Batches*/"
        "dataset19_sentinel2_batch_*.csv"
    )
)

print("Sentinel-2 files:", len(s2_files))

for file in s2_files:
    print(file)

if len(s2_files) != 10:
    raise ValueError(
        f"Expected 10 Sentinel-2 files, but found {len(s2_files)}."
    )

s2_df = pd.concat(
    [pd.read_csv(file) for file in s2_files],
    ignore_index=True
)

s2_df = s2_df[
    ["grid_id", "NDVI", "NDBI"]
].copy()

if s2_df["grid_id"].duplicated().any():
    raise ValueError("Duplicate grid IDs found in Sentinel-2 data.")

print("\nSentinel-2 shape:", s2_df.shape)
print("Unique grid IDs:", s2_df["grid_id"].nunique())
print("Missing NDVI:", s2_df["NDVI"].isna().sum())
print("Missing NDBI:", s2_df["NDBI"].isna().sum())

display(s2_df.head())

Sentinel-2 files: 10
/content/drive/MyDrive/Dataset19_Sentinel2_Batches/dataset19_sentinel2_batch_0_500.csv
/content/drive/MyDrive/Dataset19_Sentinel2_Batches/dataset19_sentinel2_batch_1000_1500.csv
/content/drive/MyDrive/Dataset19_Sentinel2_Batches/dataset19_sentinel2_batch_1500_2000.csv
/content/drive/MyDrive/Dataset19_Sentinel2_Batches/dataset19_sentinel2_batch_2000_2500.csv
/content/drive/MyDrive/Dataset19_Sentinel2_Batches/dataset19_sentinel2_batch_2500_3000.csv
/content/drive/MyDrive/Dataset19_Sentinel2_Batches/dataset19_sentinel2_batch_3000_3500.csv
/content/drive/MyDrive/Dataset19_Sentinel2_Batches/dataset19_sentinel2_batch_3500_4000.csv
/content/drive/MyDrive/Dataset19_Sentinel2_Batches/dataset19_sentinel2_batch_4000_4500.csv
/content/drive/MyDrive/Dataset19_Sentinel2_Batches/dataset19_sentinel2_batch_4500_4985.csv
/content/drive/MyDrive/Dataset19_Sentinel2_Batches/dataset19_sentinel2_batch_500_1000.csv

Sentinel-2 shape: (4985, 3)
Unique grid IDs: 4985
Missing NDVI: 0
Missing

,grid_id,NDVI,NDBI
0,3975,0.524089,-0.080509
1,3976,0.463781,-0.047096
2,4072,0.527302,-0.078790
3,4166,0.585328,-0.124289
4,4167,0.561595,-0.088098


In [ ]:
# ============================================================
# Step 12. Build Dataset19 and Merge Sentinel Features
# ============================================================

# Start with the union of LTE- and 5G NR-labelled grids
dataset19 = labelled_grid[
    ["grid_id", "geometry"]
].copy()

# Merge LTE labels
dataset19 = dataset19.merge(
    lte,
    on="grid_id",
    how="left",
    validate="one_to_one"
)

# Merge 5G NR labels
dataset19 = dataset19.merge(
    nr,
    on="grid_id",
    how="left",
    validate="one_to_one"
)

# Merge Sentinel-1 features
dataset19 = dataset19.merge(
    s1_df,
    on="grid_id",
    how="left",
    validate="one_to_one"
)

# Merge Sentinel-2 features
dataset19 = dataset19.merge(
    s2_df,
    on="grid_id",
    how="left",
    validate="one_to_one"
)

# Validate merged dataset
print("Dataset19 shape:", dataset19.shape)
print("Unique grid IDs:", dataset19["grid_id"].nunique())
print("LTE labelled grids:", dataset19["lte_signal_class"].notna().sum())
print("5G NR labelled grids:", dataset19["nr_signal_class"].notna().sum())

print("\nMissing Sentinel values:")
print(dataset19[["VV", "VH", "NDVI", "NDBI"]].isna().sum())

display(dataset19.head())

Dataset19 shape: (4985, 16)
Unique grid IDs: 4985
LTE labelled grids: 4970
5G NR labelled grids: 4965

Missing Sentinel values:
VV      0
VH      0
NDVI    0
NDBI    0
dtype: int64


,grid_id,geometry,lte_min_rsrp,lte_mean_rsrp,lte_median_rsrp,lte_point_count,lte_signal_class,nr_min_rsrp,nr_mean_rsrp,nr_median_rsrp,nr_point_count,nr_signal_class,VV,VH,NDVI,NDBI
0,3975,"POLYGON ((426000 576000, 426000 577000, 425000...",-55.64,-46.228794,-45.48,937.0,Excellent,-128.64,-89.648030,-92.37,1755.0,Good,-8.570331,-17.172422,0.524089,-0.080509
1,3976,"POLYGON ((427000 576000, 427000 577000, 426000...",-81.71,-68.405396,-69.13,1609.0,Excellent,-130.87,-92.905040,-92.43,2657.0,Good,-7.236509,-16.062169,0.463781,-0.047096
2,4072,"POLYGON ((427000 575000, 427000 576000, 426000...",-89.99,-83.527565,-84.71,1134.0,Good,-138.30,-100.509040,-96.83,2055.0,Good,-7.423318,-15.959686,0.527302,-0.078790
3,4166,"POLYGON ((424000 574000, 424000 575000, 423000...",-74.81,-64.096870,-62.38,624.0,Excellent,-128.37,-90.326965,-93.48,1207.0,Good,-10.591515,-17.102621,0.585328,-0.124289
4,4167,"POLYGON ((425000 574000, 425000 575000, 424000...",-74.29,-70.279630,-70.44,216.0,Excellent,-131.27,-91.655655,-94.24,361.0,Good,-12.357060,-19.367379,0.561595,-0.088098


In [ ]:
# ============================================================
# Step 13. Save Dataset19 Sentinel Feature Checkpoint
# ============================================================

output_dir = os.path.join(
    base_dir,
    "08_Final_Datasets"
)

csv_path = os.path.join(
    output_dir,
    "dataset19_sentinel1_sentinel2.csv"
)

gpkg_path = os.path.join(
    output_dir,
    "dataset19_sentinel1_sentinel2.gpkg"
)

# Save tabular dataset without geometry
dataset19.drop(
    columns="geometry"
).to_csv(
    csv_path,
    index=False
)

# Save spatial dataset with geometry
dataset19.to_file(
    gpkg_path,
    layer="dataset19_sentinel1_sentinel2",
    driver="GPKG"
)

print("CSV saved:")
print(csv_path)

print("\nGeoPackage saved:")
print(gpkg_path)

CSV saved:
/content/drive/MyDrive/Dissertation/Experiments/최종실험(수정)/Dataset_Build/08_Final_Datasets/dataset19_sentinel1_sentinel2.csv

GeoPackage saved:
/content/drive/MyDrive/Dissertation/Experiments/최종실험(수정)/Dataset_Build/08_Final_Datasets/dataset19_sentinel1_sentinel2.gpkg


## 5. ESA WorldCover Land-Cover Features

ESA WorldCover data are used to describe the composition of each labelled grid cell.

Pixel-frequency histograms are extracted and converted into proportional features for five selected land-cover categories:

- tree cover;
- grassland;
- cropland;
- built-up land; and
- permanent water bodies.

※ Related dissertation sections
 - Section 2.6 (Public Geospatial Data as Predictors)
 - Section 3.1 (Research Approach—Data Preparation)
 - Section 4.1.1 (Baseline Dataset Construction).

In [ ]:
# ============================================================
# Step 14. Prepare ESA WorldCover
# ============================================================

worldcover = (
    ee.ImageCollection("ESA/WorldCover/v200")
    .first()
    .select("Map")
)

print("WorldCover ready")

WorldCover ready


In [ ]:
# ============================================================
# Step 15. Export ESA WorldCover by Batch
# ============================================================

batch_size = 500
n = len(grid_wgs84)

for start in range(0, n, batch_size):

    end = min(start + batch_size, n)

    batch_gdf = grid_wgs84.iloc[start:end][
        ["grid_id", "geometry"]
    ].copy()

    batch_geojson = json.loads(
        batch_gdf.to_json()
    )

    grids_batch = ee.FeatureCollection(batch_geojson)

    # Count pixels belonging to each land-cover class
    result = worldcover.reduceRegions(
        collection=grids_batch,
        reducer=ee.Reducer.frequencyHistogram(),
        scale=10,
        tileScale=16
    ).select(
        ["grid_id", "histogram"]
    )

    task = ee.batch.Export.table.toDrive(
        collection=result,
        description=f"dataset19_worldcover_batch_{start}_{end}",
        folder="Dataset19_WorldCover_Batches",
        fileNamePrefix=f"dataset19_worldcover_batch_{start}_{end}",
        fileFormat="CSV"
    )

    task.start()

    print(f"Started batch {start:,}-{end:,}")

print("\nAll 10 WorldCover export tasks submitted.")

Started batch 0-500
Started batch 500-1,000
Started batch 1,000-1,500
Started batch 1,500-2,000
Started batch 2,000-2,500
Started batch 2,500-3,000
Started batch 3,000-3,500
Started batch 3,500-4,000
Started batch 4,000-4,500
Started batch 4,500-4,985

All 10 WorldCover export tasks submitted.


## 6. Population and Night-Time-Light Features

WorldPop data are used to calculate the total population within each labelled grid cell.

VIIRS monthly night-time-light imagery from August to October 2025 is used to calculate median radiance and extract the mean value for each grid. Night-time light provides a broad indicator of human activity and urban development.

※Related dissertation sections
 - Section 2.6 (Public Geospatial Data as Predictors)
 - Section 3.1 (Research Approach—Data Preparation)
 - Section 4.1.1 (Baseline Dataset Construction).

In [ ]:
# ============================================================
# Step 16. Prepare Population
# ============================================================

population = (
    ee.ImageCollection("WorldPop/GP/100m/pop")
    .filterDate("2020-01-01", "2021-01-01")
    .filter(ee.Filter.eq("country", "GBR"))
    .first()
    .select("population")
)

print("Population ready")

Population ready


In [ ]:
# ============================================================
# Step 17. Export Population by Batch
# ============================================================

batch_size = 500
n = len(grid_wgs84)

for start in range(0, n, batch_size):

    end = min(start + batch_size, n)

    batch_gdf = grid_wgs84.iloc[start:end][
        ["grid_id", "geometry"]
    ].copy()

    batch_geojson = json.loads(
        batch_gdf.to_json()
    )

    grids_batch = ee.FeatureCollection(batch_geojson)

    # Calculate total population within each 1 km grid
    result = population.reduceRegions(
        collection=grids_batch,
        reducer=ee.Reducer.sum(),
        scale=100,
        tileScale=16
    ).select(
        ["grid_id", "sum"]
    )

    task = ee.batch.Export.table.toDrive(
        collection=result,
        description=f"dataset19_population_batch_{start}_{end}",
        folder="Dataset19_Population_Batches",
        fileNamePrefix=f"dataset19_population_batch_{start}_{end}",
        fileFormat="CSV"
    )

    task.start()

    print(f"Started batch {start:,}-{end:,}")

print("\nAll 10 population export tasks submitted.")

Started batch 0-500
Started batch 500-1,000
Started batch 1,000-1,500
Started batch 1,500-2,000
Started batch 2,000-2,500
Started batch 2,500-3,000
Started batch 3,000-3,500
Started batch 3,500-4,000
Started batch 4,000-4,500
Started batch 4,500-4,985

All 10 population export tasks submitted.


In [ ]:
# ============================================================
# Step 18. Prepare Nightlight
# ============================================================

nightlight = (
    ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG")
    .filterDate("2025-08-01", "2025-11-01")
    .select("avg_rad")
    .median()
)

print("Nightlight ready")

Nightlight ready


In [ ]:
# ============================================================
# Step 19. Export Nightlight by Batch
# ============================================================

batch_size = 500
n = len(grid_wgs84)

for start in range(0, n, batch_size):

    end = min(start + batch_size, n)

    batch_gdf = grid_wgs84.iloc[start:end][
        ["grid_id", "geometry"]
    ].copy()

    batch_geojson = json.loads(
        batch_gdf.to_json()
    )

    grids_batch = ee.FeatureCollection(batch_geojson)

    # Calculate mean night-time radiance within each grid
    result = nightlight.reduceRegions(
        collection=grids_batch,
        reducer=ee.Reducer.mean(),
        scale=500,
        tileScale=16
    ).select(
        ["grid_id", "mean"]
    )

    task = ee.batch.Export.table.toDrive(
        collection=result,
        description=f"dataset19_nightlight_batch_{start}_{end}",
        folder="Dataset19_Nightlight_Batches",
        fileNamePrefix=f"dataset19_nightlight_batch_{start}_{end}",
        fileFormat="CSV"
    )

    task.start()

    print(f"Started batch {start:,}-{end:,}")

print("\nAll 10 nightlight export tasks submitted.")

Started batch 0-500
Started batch 500-1,000
Started batch 1,000-1,500
Started batch 1,500-2,000
Started batch 2,000-2,500
Started batch 2,500-3,000
Started batch 3,000-3,500
Started batch 3,500-4,000
Started batch 4,000-4,500
Started batch 4,500-4,985

All 10 nightlight export tasks submitted.


## 7. Feature Integration and Validation

The WorldCover, population, and night-time-light batch outputs are combined and merged with the Sentinel features and signal labels using 'grid_id'.

The completed environmental dataset contains 11 predictors:

- Sentinel-1: VV and VH;
- Sentinel-2: NDVI and NDBI;
- WorldCover: five land-cover proportions;
- WorldPop: population; and
- VIIRS: night-time light.

The dataset is exported as both CSV and GeoPackage files. The final checks confirm grid-ID uniqueness, label counts, feature completeness, and descriptive statistics.

※Related dissertation sections
 - Section 3.1 (Research Approach—Data Preparation)
 - Section 4.1.1 (Baseline Dataset Construction)
 - Appendix C (Reproducibility and Predictor Definitions).

In [ ]:
# ============================================================
# Step 20. Merge WorldCover, Population, and Nightlight Batches
# ============================================================

import glob

# ------------------------------------------------------------
# 1. WorldCover
# ------------------------------------------------------------

wc_files = sorted(
    glob.glob(
        "/content/drive/MyDrive/"
        "Dataset19_WorldCover_Batches*/"
        "dataset19_worldcover_batch_*.csv"
    )
)

print("WorldCover files:", len(wc_files))

if len(wc_files) != 10:
    raise ValueError(
        f"Expected 10 WorldCover files, but found {len(wc_files)}."
    )

wc = pd.concat(
    [pd.read_csv(file) for file in wc_files],
    ignore_index=True
)

wc = wc[["grid_id", "histogram"]].copy()

# ------------------------------------------------------------
# 2. Population
# ------------------------------------------------------------

pop_files = sorted(
    glob.glob(
        "/content/drive/MyDrive/"
        "Dataset19_Population_Batches*/"
        "dataset19_population_batch_*.csv"
    )
)

print("Population files:", len(pop_files))

if len(pop_files) != 10:
    raise ValueError(
        f"Expected 10 population files, but found {len(pop_files)}."
    )

pop = pd.concat(
    [pd.read_csv(file) for file in pop_files],
    ignore_index=True
)

pop = pop[
    ["grid_id", "sum"]
].rename(
    columns={"sum": "population"}
)

# ------------------------------------------------------------
# 3. Nightlight
# ------------------------------------------------------------

nl_files = sorted(
    glob.glob(
        "/content/drive/MyDrive/"
        "Dataset19_Nightlight_Batches*/"
        "dataset19_nightlight_batch_*.csv"
    )
)

print("Nightlight files:", len(nl_files))

if len(nl_files) != 10:
    raise ValueError(
        f"Expected 10 nightlight files, but found {len(nl_files)}."
    )

nl = pd.concat(
    [pd.read_csv(file) for file in nl_files],
    ignore_index=True
)

nl = nl[
    ["grid_id", "mean"]
].rename(
    columns={"mean": "nightlight"}
)

# Validate results
print("\nWorldCover shape:", wc.shape)
print("Population shape:", pop.shape)
print("Nightlight shape:", nl.shape)

print("\nUnique grid IDs:")
print("WorldCover:", wc["grid_id"].nunique())
print("Population:", pop["grid_id"].nunique())
print("Nightlight:", nl["grid_id"].nunique())

WorldCover files: 10
Population files: 10
Nightlight files: 10

WorldCover shape: (4985, 2)
Population shape: (4985, 2)
Nightlight shape: (4985, 2)

Unique grid IDs:
WorldCover: 4985
Population: 4985
Nightlight: 4985


In [ ]:
# ============================================================
# Step 21. Merge Population and Nightlight Features
# ============================================================

dataset19 = dataset19.merge(
    pop,
    on="grid_id",
    how="left",
    validate="one_to_one"
)

dataset19 = dataset19.merge(
    nl,
    on="grid_id",
    how="left",
    validate="one_to_one"
)

print("Dataset19 shape:", dataset19.shape)
print("\nMissing values:")
print(dataset19[["population", "nightlight"]].isna().sum())

display(dataset19.head())

Dataset19 shape: (4985, 18)

Missing values:
population    0
nightlight    0
dtype: int64


,grid_id,geometry,lte_min_rsrp,lte_mean_rsrp,lte_median_rsrp,lte_point_count,lte_signal_class,nr_min_rsrp,nr_mean_rsrp,nr_median_rsrp,nr_point_count,nr_signal_class,VV,VH,NDVI,NDBI,population,nightlight
0,3975,"POLYGON ((426000 576000, 426000 577000, 425000...",-55.64,-46.228794,-45.48,937.0,Excellent,-128.64,-89.648030,-92.37,1755.0,Good,-8.570331,-17.172422,0.524089,-0.080509,1594.242380,8.347392
1,3976,"POLYGON ((427000 576000, 427000 577000, 426000...",-81.71,-68.405396,-69.13,1609.0,Excellent,-130.87,-92.905040,-92.43,2657.0,Good,-7.236509,-16.062169,0.463781,-0.047096,3061.673353,14.387132
2,4072,"POLYGON ((427000 575000, 427000 576000, 426000...",-89.99,-83.527565,-84.71,1134.0,Good,-138.30,-100.509040,-96.83,2055.0,Good,-7.423318,-15.959686,0.527302,-0.078790,2471.479708,7.935724
3,4166,"POLYGON ((424000 574000, 424000 575000, 423000...",-74.81,-64.096870,-62.38,624.0,Excellent,-128.37,-90.326965,-93.48,1207.0,Good,-10.591515,-17.102621,0.585328,-0.124289,173.731904,5.853407
4,4167,"POLYGON ((425000 574000, 425000 575000, 424000...",-74.29,-70.279630,-70.44,216.0,Excellent,-131.27,-91.655655,-94.24,361.0,Good,-12.357060,-19.367379,0.561595,-0.088098,40.148841,3.391786


In [ ]:
# ============================================================
# Step 22. Convert WorldCover Histograms to Land-Cover Ratios
# ============================================================

import ast
import re

# Parse Earth Engine histogram strings
def parse_wc_hist(value):

    if pd.isna(value):
        return {}

    # Convert format such as {10=2500, 20=500} to Python dictionary
    cleaned = re.sub(
        r"(\d+(?:\.\d+)?)=",
        r'"\1":',
        str(value)
    )

    return ast.literal_eval(cleaned)


# Calculate the proportion of a land-cover class
def get_ratio(histogram, code):

    total = sum(histogram.values())

    if total == 0:
        return np.nan

    value = (
        histogram.get(str(code), 0)
        + histogram.get(f"{code}.0", 0)
        + histogram.get(code, 0)
    )

    return value / total


# Parse histograms
wc["hist_dict"] = wc["histogram"].apply(
    parse_wc_hist
)

# Create selected land-cover ratio features
wc_features = pd.DataFrame({
    "grid_id": wc["grid_id"],
    "tree_ratio": wc["hist_dict"].apply(
        lambda d: get_ratio(d, 10)
    ),
    "grass_ratio": wc["hist_dict"].apply(
        lambda d: get_ratio(d, 30)
    ),
    "crop_ratio": wc["hist_dict"].apply(
        lambda d: get_ratio(d, 40)
    ),
    "builtup_ratio": wc["hist_dict"].apply(
        lambda d: get_ratio(d, 50)
    ),
    "water_ratio": wc["hist_dict"].apply(
        lambda d: get_ratio(d, 80)
    )
})

# Validate WorldCover features
print("WorldCover feature shape:", wc_features.shape)
print("\nMissing values:")
print(wc_features.isna().sum())

# Merge with Dataset19
dataset19 = dataset19.merge(
    wc_features,
    on="grid_id",
    how="left",
    validate="one_to_one"
)

print("\nDataset19 shape:", dataset19.shape)

display(wc_features.head())

WorldCover feature shape: (4985, 6)

Missing values:
grid_id          0
tree_ratio       0
grass_ratio      0
crop_ratio       0
builtup_ratio    0
water_ratio      0
dtype: int64

Dataset19 shape: (4985, 23)


,grid_id,tree_ratio,grass_ratio,crop_ratio,builtup_ratio,water_ratio
0,3975,0.195237,0.371513,0.235360,0.197890,0.000000
1,3976,0.222356,0.254962,0.000000,0.522511,0.000000
2,4072,0.237174,0.341157,0.000000,0.421669,0.000000
3,4166,0.357394,0.422191,0.067523,0.152720,0.000057
4,4167,0.124595,0.423217,0.424147,0.027295,0.000746


In [ ]:
# ============================================================
# Step 23. Save Dataset19 Environmental Features
# ============================================================

output_dir = os.path.join(
    base_dir,
    "08_Final_Datasets"
)

csv_path = os.path.join(
    output_dir,
    "dataset19_s1_s2_worldcover_population_nightlight.csv"
)

gpkg_path = os.path.join(
    output_dir,
    "dataset19_s1_s2_worldcover_population_nightlight.gpkg"
)

# Save tabular data without geometry
dataset19.drop(
    columns="geometry"
).to_csv(
    csv_path,
    index=False
)

# Save spatial data with geometry
dataset19.to_file(
    gpkg_path,
    layer="dataset19_environmental_features",
    driver="GPKG"
)

print("Dataset shape:", dataset19.shape)

print("\nCSV saved:")
print(csv_path)

print("\nGeoPackage saved:")
print(gpkg_path)

Dataset shape: (4985, 23)

CSV saved:
/content/drive/MyDrive/Dissertation/Experiments/최종실험(수정)/Dataset_Build/08_Final_Datasets/dataset19_s1_s2_worldcover_population_nightlight.csv

GeoPackage saved:
/content/drive/MyDrive/Dissertation/Experiments/최종실험(수정)/Dataset_Build/08_Final_Datasets/dataset19_s1_s2_worldcover_population_nightlight.gpkg


In [ ]:
# ============================================================
# Step 24. Validate Saved Dataset19
# ============================================================

check_path = os.path.join(
    base_dir,
    "08_Final_Datasets",
    "dataset19_s1_s2_worldcover_population_nightlight.csv"
)

df = pd.read_csv(check_path)

feature_cols = [
    "VV", "VH", "NDVI", "NDBI",
    "tree_ratio", "grass_ratio", "crop_ratio",
    "builtup_ratio", "water_ratio",
    "population", "nightlight"
]

print("Shape:", df.shape)
print("Unique grid IDs:", df["grid_id"].nunique())
print("LTE labelled:", df["lte_signal_class"].notna().sum())
print("5G NR labelled:", df["nr_signal_class"].notna().sum())

print("\nFeature missing values:")
print(df[feature_cols].isna().sum())

print("\nFeature summary:")
display(df[feature_cols].describe())

Shape: (4985, 22)
Unique grid IDs: 4985
LTE labelled: 4970
5G NR labelled: 4965

Feature missing values:
VV               0
VH               0
NDVI             0
NDBI             0
tree_ratio       0
grass_ratio      0
crop_ratio       0
builtup_ratio    0
water_ratio      0
population       0
nightlight       0
dtype: int64

Feature summary:


,VV,VH,NDVI,NDBI,tree_ratio,grass_ratio,crop_ratio,builtup_ratio,water_ratio,population,nightlight
count,4985.000000,4985.000000,4985.000000,4985.000000,4985.000000,4985.000000,4985.000000,4985.000000,4985.000000,4985.000000,4985.000000
mean,-9.576405,-16.714619,0.540825,-0.118656,0.275933,0.289763,0.158411,0.262703,0.010897,1603.552907,11.963639
std,2.223489,2.161225,0.132898,0.086152,0.163199,0.224181,0.234504,0.239595,0.055484,2042.129078,13.119879
min,-19.210204,-28.396603,-0.155942,-0.818662,0.000000,0.000118,0.000000,0.000000,0.000000,0.142489,0.491805
25%,-11.238779,-18.396803,0.454380,-0.171163,0.150720,0.110485,0.000008,0.051967,0.000000,68.685032,2.592241
50%,-9.923507,-16.809941,0.539920,-0.111487,0.261865,0.229203,0.015501,0.177311,0.000000,575.634055,7.708562
75%,-7.998521,-15.045517,0.641018,-0.061708,0.385123,0.417801,0.258828,0.453836,0.000468,2729.616174,16.382044
max,2.423742,-4.961346,0.856636,0.147158,0.915909,0.999165,0.984214,0.980152,0.973215,15462.499642,109.112625
